In [18]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

In [19]:
documents = PyPDFLoader("../../arquivos/exemplo_rh.pdf").load()
print(f"Documento carregado. Total de documents: {len(documents)}")

Documento carregado. Total de documents: 9


In [21]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len
)

docs = text_splitter.split_documents(documents)
print(f"Total de chunks criados: {len(docs)}")

Total de chunks criados: 19


In [22]:

persist_directory = "./chroma_db"
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

if os.path.exists(persist_directory):
    vector_store = Chroma(persist_directory=persist_directory, embedding_function=embeddings)
    print("Já criada")
else:
    vector_store = Chroma.from_documents(docs, embeddings, persist_directory=persist_directory)
    print(f"Não existe ainda. Criando Vector Store")

print("Vector Store criada com sucesso!")

Já criada
Vector Store criada com sucesso!


In [31]:
retriever = vector_store.as_retriever(search_kwargs={"k":2})
# llm = init_chat_model("google_genai:gemini-flash-latest", temperature=0)  # 503 por sobrecarga
# llm = init_chat_model("groq:openai/gpt-oss-120b", temperature=0)
llm = init_chat_model("google_genai:gemini-3.1-flash-lite", temperature=0, max_retries=5)

In [32]:
pergunta = input("Sou o oráculo do RH. Faça sua pergunta: ")
similar_docs = retriever.invoke(pergunta)
contexto = "\n\n".join([doc.page_content for doc in similar_docs])
resposta = llm.invoke(f"Com base no seguinte contexto \n\n {contexto}\n\n responda a seguinte pergunta: {pergunta}")

In [34]:
print(resposta.text)

Com base no texto fornecido, os benefícios oferecidos pela empresa são:

*   **Vale-Refeição:** R$ 35,00 por dia útil trabalhado.
*   **Vale-Transporte:** Conforme a legislação, com dedução de 6% do salário bruto.
*   **Plano de Saúde:** Cobertura médica e hospitalar, com a empresa arcando com 70% do valor.
*   **Plano Odontológico:** Opcional, com desconto em folha.
*   **Seguro de Vida:** Cobertura de 36x o salário, sem custo para o colaborador.
*   **EPI (Equipamento de Proteção Individual):** Fornecimento integral e sem custo para colaboradores de obra.
*   **Participação nos Resultados (PLR):** Pagamento anual condicionado ao atingimento de metas.
*   **Adiantamento Salarial:** Possibilidade de solicitar até 40% do salário líquido uma vez por mês (solicitação até o dia 10, crédito no dia 15).
